# Часть A. Подготовка: доступы, машина, инструменты (Google Colab)

<a href="https://colab.research.google.com/github/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/notebooks/A_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Блоки 1–6 «Плана кода» для связки Windows + Google Colab. Срок по плану: 25 сентября — 5 октября.

- **Windows** — код пишется в IDE и отправляется в GitHub (`git push` с ноутбука уже работает).
- **Colab** — всё, что считается на GPU. Блокноты открываются из GitHub: File → Open notebook → GitHub.
- **Google Drive** — всё, что нельзя потерять (аналог большого диска `<BIG>` из плана). На Windows эти файлы видны через Google Drive для компьютера.

| В плане (сервер) | Здесь |
|---|---|
| большой диск `<BIG>` | Google Drive, папка `MyDrive/unlearning_data` |
| `~/.bashrc` | ячейка «Старт сессии»: машина Colab каждый раз новая |
| `hf auth login` | секрет `HF_TOKEN` в Colab Secrets (🔑 на левой панели) |
| SSH-ключ и `git push` с сервера | `git push` с Windows; Colab только читает репозиторий |
| SSH, VS Code Remote, tmux | вкладка браузера с блокнотом (A3) |
| apt, Miniforge, conda | ставить ничего не нужно; окружения uv — в части B |

## A1. Аккаунты и доступы [один раз, руками]

1. **Hugging Face.** Зарегистрироваться на huggingface.co → Settings → Access Tokens → Create new token, тип **Read**. В Colab: 🔑 Secrets → Add new secret → имя `HF_TOKEN`, значение — токен → включить Notebook access. Токен — это пароль: в ячейки и в git он не попадает.
2. **Лицензии Llama — запасной путь.** Основной путь плана берёт токенизатор из моделей `open-unlearning/tofu_*`, и доступ к `meta-llama` ему не нужен. Для запасного пути подать заявки на страницах `meta-llama/Llama-3.2-1B-Instruct`, `meta-llama/Llama-3.2-3B-Instruct` и `meta-llama/Llama-3.1-8B-Instruct`. Статус заявок — на huggingface.co/settings/gated-repos.
3. **GitHub.** Push идёт с Windows и уже работает. Colab только читает публичный репозиторий, поэтому токен GitHub ему не нужен. Он понадобится, только если репозиторий станет приватным или вы решите пушить прямо из Colab.
4. **Colab.** Бесплатная T4 для обучения не годится: у неё compute capability 7.5, а bf16 и FlashAttention-2 в конфигах OpenUnlearning требуют 8.0 и выше. Платные тарифы дают L4, A100 и H100 (когда свободны). В Pro вкладку с блокнотом нужно держать открытой, в Pro+ сессия продолжается и с закрытой вкладкой. Можно начать с Pro и перейти на Pro+, если долгие запуски начнут обрываться; точные условия — на странице тарифов Colab. A100 и H100 расходуют compute units в разы быстрее L4, поэтому отладку лучше вести на L4. Для 8B (апрель) нужно 2 × 80 ГБ — в Colab это невозможно, и план для неё предусматривает аренду.
5. **Google Drive.** Бесплатных 15 ГБ хватит на части A–D (чекпоинт 1B весит 2,5 ГБ). К ноябрю, когда пойдут чекпоинты 3B по 6,4 ГБ, скорее всего понадобится Google One на 100–200 ГБ.

**Готово, когда:** секрет `HF_TOKEN` добавлен, и проверка A1 ниже печатает ваше имя; заявки на Llama поданы (по желанию); подписка Colab оформлена.

---
## Старт сессии [каждая сессия]

Одна ячейка заменяет `~/.bashrc` и блок 6 плана. Она же будет первой в блокнотах B, C и D.

```
Google Drive — постоянно (аналог <BIG>)     Машина Colab — быстро, стирается после сессии
MyDrive/unlearning_data/                    /content/fast/
├── saves/        чекпоинты и оценки        ├── hf_home/   кэш Hugging Face (HF_HOME)
├── results_raw/  сырые журналы атак        └── models/    модели (MODELS), часть C
├── logs/         логи долгих задач
└── journal.md    лабораторный журнал (блок 18)
```

Модели занимают 85–120 ГБ и каждую сессию за минуты скачиваются заново, поэтому лежат на быстром диске машины. Всё уникальное сразу пишется на Drive.

In [ ]:
# ===== Старт сессии: выполнять после каждого подключения к Colab =====
import os

from google.colab import drive, userdata

DRIVE_ROOT = "/content/drive/MyDrive/unlearning_data"  # постоянное хранилище (аналог <BIG> из плана)
FAST_ROOT = "/content/fast"                             # быстрый диск машины, стирается после сессии

# 1. Google Drive: в первый раз Colab попросит разрешение
drive.mount("/content/drive")

# 2. Папки: уже существующие не трогаются
for folder in [DRIVE_ROOT + "/saves", DRIVE_ROOT + "/results_raw", DRIVE_ROOT + "/logs",
               FAST_ROOT + "/hf_home", FAST_ROOT + "/models"]:
    os.makedirs(folder, exist_ok=True)

# 3. Переменные окружения: их видят и Python, и все !-команды
os.environ["BIG"] = DRIVE_ROOT
os.environ["HF_HOME"] = FAST_ROOT + "/hf_home"   # задаётся до первого импорта библиотек Hugging Face
os.environ["MODELS"] = FAST_ROOT + "/models"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # без предупреждений токенизаторов в подпроцессах
os.environ["PYTHONUNBUFFERED"] = "1"             # вывод Python сразу попадает в лог
!echo "BIG=$BIG  HF_HOME=$HF_HOME  MODELS=$MODELS"

# 4. Какая GPU досталась: имя, память, compute capability
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

# 5. Токен Hugging Face из Colab Secrets. Если секрета нет, Colab выдаст ошибку — добавьте его (A1)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Старт сессии выполнен")

## A1 — проверка [один раз]

Замена `hf auth whoami` из плана: токен действует, если печатается ваше имя на Hugging Face.

In [ ]:
from huggingface_hub import whoami

print("Hugging Face: вы вошли как", whoami()["name"])

---
## A2. Машина [один раз на каждый тип GPU]

| GPU в Colab | Память | cc | Для чего |
|---|---|---|---|
| T4 | 16 ГБ | 7.5 | только знакомство; обучение по конфигам OpenUnlearning — нет |
| L4 | 24 ГБ | 8.9 | unlearning 1B и отладка; атаки на 1B и 3B с атакующим в AWQ |
| A100 | обычно 40 ГБ | 8.0 | unlearning 1B с запасом; для 3B мало (нужно от 48 ГБ) |
| H100 | 80 ГБ | 9.0 | unlearning 3B; атаки с несжатым атакующим |

**Как читать `nvidia-smi`.** «CUDA Version» справа вверху — максимальная версия CUDA, которую поддерживает драйвер (нужна 12.x или выше); ставить CUDA Toolkit отдельно не нужно. В нижней таблице — процессы на GPU.

In [ ]:
!nvidia-smi

In [ ]:
# Характеристики машины — в журнал на Drive (блок 18). Одна запись на тип GPU:
# при повторном запуске запись добавится ещё раз, лишнюю можно удалить в journal.md.
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

import psutil

gpu = !nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader
cuda = !nvidia-smi | grep -o "CUDA Version: [0-9.]*"
gpu_text = " ".join(gpu)
cuda_text = " ".join(cuda)
ram_gb = round(psutil.virtual_memory().total / 1e9)
disk = shutil.disk_usage("/content")
today = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y-%m-%d")

entry = f"""
## {today} — Проверка машины (Colab)
- GPU (имя, память, compute capability, драйвер): {gpu_text}
- {cuda_text}
- CPU: {os.cpu_count()} ядер; ОЗУ: {ram_gb} ГБ
- Диск машины: свободно {round(disk.free / 1e9)} из {round(disk.total / 1e9)} ГБ
- Наблюдения: …
"""
with open(DRIVE_ROOT + "/journal.md", "a", encoding="utf-8") as f:
    f.write(entry)
print(entry)

**Готово, когда:** в `journal.md` на Drive есть запись с GPU, драйвером, CUDA, CPU, ОЗУ и диском. Если досталась другая GPU, выполните ячейку ещё раз.

---
## A3. Подключение и долгие задачи [прочитать]

- **Подключение** — это вкладка браузера. GPU выбирается в Runtime → Change runtime type; для 1B и отладки хватит L4.
- **Машина временная.** При отключении пропадают всё содержимое `/content` и все процессы. Поэтому чекпоинты, логи и результаты сразу пишутся на Drive, а каждый долгий шаг должен уметь продолжить работу с места обрыва (в плане за это отвечают маркеры `DONE`, блок 23).
- **tmux не нужен.** Задачи на минуты идут прямо в ячейке. Серверы vLLM и длинные очереди запускаются в фоне командой `nohup … > лог 2>&1 &` — это появится в части D, где впервые понадобится.
- **TensorBoard** — прямо в блокноте: `%load_ext tensorboard`, затем `%tensorboard --logdir <папка>`.
- **Runtime → Restart session** перезапускает только Python: файлы и фоновые процессы остаются (забытый сервер vLLM продолжит занимать память GPU — проверьте `!nvidia-smi`). **Disconnect and delete runtime** выдаёт чистую машину.
- **rsync не нужен:** файлы с Drive видны на Windows через Google Drive для компьютера.

## A4. Шпаргалка по Colab

| Команда | Что делает |
|---|---|
| `!команда` | команда оболочки; каждая `!`-строка — отдельный процесс |
| `x = !команда` | вывод команды — в список строк Python |
| `%cd папка` | сменить папку надолго (`!cd` действует только внутри своей строки) |
| `os.environ["X"] = "1"` | переменная окружения для следующих `!`-команд (`!export` не сохраняется) |
| `!echo {x}`, `!echo $X` | подставить переменную Python; переменную окружения |
| `%%writefile файл` | в первой строке ячейки: записать ячейку в файл |
| `!tail -n 30 лог` | конец лога — первое, что смотреть при ошибке |
| `!ps aux \| grep python`, `!kill PID` | найти процесс и остановить его |
| `!df -h /content /content/drive` | свободное место на машине и на Drive |

Команды git и терминала на Windows — в таблицах блока 4 плана (Git Bash или терминал IDE).

## A5. Базовое ПО

В Colab уже есть git, curl, wget, компилятор и драйвер NVIDIA, поэтому ставить ничего не нужно. На Windows достаточно git — он уже настроен. Окружения `unl` и `atk` соберём в части B через uv вместо conda.

**Дальше — часть B:** окружения `unl` (OpenUnlearning) и `atk` (vLLM, оценщик).